In [4]:
%pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 39.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 126.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]
Note: you may need to restart the kernel to use updated packages.


In [11]:
import polars as pl
import numpy as np
import pandas as pd
from datetime import date
import gc

In [ ]:
file_path = 'data_ready/df_2025_valid_only.parquet'
lazy_df = pl.scan_parquet(file_path)
drop_columns = ['serial_number', 'model']
lazy_df = lazy_df.drop(drop_columns)

print('Splitting chronologically...')

train_set = lazy_df.filter(pl.col('date') < date(2025, 10, 1))
test_set = lazy_df.filter(pl.col('date') >= date(2025, 10, 1))

print('Downsampling healthy drives in training set...')
train_fail = train_set.filter(pl.col('failure') == 1)
train_healthy = train_set.filter(pl.col('failure') == 0).gather_every(20)

test_fail = test_set.filter(pl.col('failure') == 1)
test_healthy = test_set.filter(pl.col('failure') == 0).gather_every(20)

lazy_train = pl.concat([train_fail, train_healthy])
lazy_test = pl.concat([test_fail, test_healthy])

lazy_train.sink_parquet("data_ready/train_split.parquet")

lazy_test.sink_parquet("data_ready/test_split.parquet")

print("Loading optimized datasets into memory...")
df_train = pl.read_parquet("data_ready/train_split.parquet")
df_test = pl.read_parquet("data_ready/test_split.parquet")

print("Applying final memory-safe downsample...")
train_fail = df_train.filter(pl.col("failure") == 1)
train_healthy = df_train.filter(pl.col("failure") == 0).sample(fraction=0.10, seed=6740)
df_train = pl.concat([train_fail, train_healthy])

test_fail = df_test.filter(pl.col("failure") == 1)
test_healthy = df_test.filter(pl.col("failure") == 0).sample(fraction=0.10, seed=6740)
df_test = pl.concat([test_fail, test_healthy])

print(f"NEW Train Shape: {df_train.shape} | NEW Test Shape: {df_test.shape}")

print("Handling Nulls...")
df_train = df_train.fill_null(0)
df_test = df_test.fill_null(0)

drop_cols = ['date', 'failure'] + [col for col in df_train.columns if 'normalized' in col.lower()]

y_train = df_train['failure'].to_pandas()
y_test = df_test['failure'].to_pandas()

X_train_pl = df_train.drop(drop_cols)
X_test_pl = df_test.drop(drop_cols)

del df_train
del df_test
gc.collect()

print("Converting X matrices to Pandas...")
X_train = X_train_pl.to_pandas()
X_test = X_test_pl.to_pandas()

del X_train_pl
del X_test_pl
gc.collect()

print("Success! Matrices are ready for Scikit-Learn.")

Splitting chronologically...
Downsampling healthy drives in training set...
Loading optimized datasets into memory...


In [10]:
X_train.head(5)

,capacity_bytes,smart_1_raw,smart_5_raw,smart_9_raw,smart_194_raw,smart_197_raw,smart_1_normalized,smart_2_normalized,smart_2_raw,smart_3_normalized,...,cluster_id,pod_slot_num,smart_82_normalized,smart_82_raw,smart_27_normalized,smart_27_raw,smart_211_normalized,smart_211_raw,smart_212_normalized,smart_212_raw
0,8001563222016,111314472.0,22513.0,72366.0,38.0,0.0,60.0,0.0,0.0,84.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,8001563222016,136209792.0,11176.0,65025.0,42.0,0.0,53.0,0.0,0.0,88.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,12000138625024,40867320.0,0.0,33885.0,30.0,0.0,76.0,0.0,0.0,91.0,...,0,50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,12000138625024,0.0,10.0,49440.0,29.0,100.0,100.0,100.0,0.0,157.0,...,0,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,16000900661248,0.0,0.0,19241.0,39.0,0.0,100.0,135.0,100.0,84.0,...,0,33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
